# 🎬 YouTube Shorts ワンクリック自動生成

## 使い方（たったこれだけ！）
1. **セル1**でAPIキーを入力（初回のみ）
2. **セル2**でテーマを入力
3. **「ランタイム」→「すべてのセルを実行」** をクリック
4. 10〜15分待つと動画が完成します ✅

---

In [ ]:
# ════════════════════════════════════════════════════════
# セル1: APIキー設定（初回だけ入力。次回からは自動読込）
# ════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import os, json

CONFIG_PATH = '/content/drive/MyDrive/YouTube_Production/.config.json'
os.makedirs('/content/drive/MyDrive/YouTube_Production', exist_ok=True)

# 保存済みキーを読み込む
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        _cfg = json.load(f)
    GEMINI_API_KEY = _cfg.get('GEMINI_API_KEY', '')
    PEXELS_API_KEY = _cfg.get('PEXELS_API_KEY', '')
else:
    GEMINI_API_KEY = ''
    PEXELS_API_KEY = ''

# 未設定の場合だけ入力を求める
if not GEMINI_API_KEY:
    GEMINI_API_KEY = input('🔑 Gemini APIキーを入力してください: ').strip()
if not PEXELS_API_KEY:
    PEXELS_API_KEY = input('🔑 Pexels APIキーを入力してください（無料: pexels.com/api）: ').strip()

# ドライブに保存（次回から入力不要）
with open(CONFIG_PATH, 'w') as f:
    json.dump({'GEMINI_API_KEY': GEMINI_API_KEY, 'PEXELS_API_KEY': PEXELS_API_KEY}, f)

print('✅ APIキー設定完了')

In [ ]:
# ════════════════════════════════════════════════════════
# セル2: テーマと秒数を設定（ここだけ毎回変える）
# ════════════════════════════════════════════════════════

THEME    = '筋トレ初心者が1ヶ月で変わる方法'  # ← ここにテーマを書く
DURATION = 45                                  # ← 動画の長さ（30〜60秒）

print(f'テーマ : {THEME}')
print(f'目標尺 : {DURATION}秒')

In [ ]:
# ════════════════════════════════════════════════════════
# セル3: 必要なツールをインストール（自動）
# ════════════════════════════════════════════════════════

!pip install -q gtts requests
!apt-get install -q -y ffmpeg
print('✅ ツールのインストール完了')

In [ ]:
# ════════════════════════════════════════════════════════
# セル4: Gemini APIで台本を自動生成
# ════════════════════════════════════════════════════════

import requests as _req
import json, re

GEMINI_URL = 'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent'

def call_gemini(prompt):
    res = _req.post(
        f'{GEMINI_URL}?key={GEMINI_API_KEY}',
        json={
            'contents': [{'parts': [{'text': prompt}]}],
            'generationConfig': {'temperature': 0.8, 'maxOutputTokens': 4096}
        },
        timeout=120
    )
    if res.status_code != 200:
        raise RuntimeError(f'Gemini APIエラー ({res.status_code}): {res.text[:300]}')
    return res.json()['candidates'][0]['content']['parts'][0]['text']

SCENE_COUNT = 4 if DURATION <= 35 else 5 if DURATION <= 50 else 6

print('[1/5] YouTubeトレンドを分析中...')
trend_prompt = f"""
あなたはYouTubeショート動画のトレンドアナリストです。
テーマ「{THEME}」（{DURATION}秒）について、バズりやすいフック・構成・差別化のコツを教えてください。
台本制作に直接使える指針を簡潔にまとめてください。
"""
trend = call_gemini(trend_prompt)
print('  ✓ トレンド分析完了')

print('[2/5] 台本を生成中...')
script_prompt = f"""
あなたはYouTubeショート動画の台本専門ライターです。

【テーマ】{THEME}
【目標尺】{DURATION}秒（{SCENE_COUNT}シーン構成）
【トレンド分析】{trend}

以下のフォーマットで必ず出力してください。

[台本]
シーン1（フック）:
（セリフ）

シーン2（問題提起）:
（セリフ）

シーン3（本題①）:
（セリフ）

シーン4（本題②）:
（セリフ）

{'シーン5（まとめ）:\n（セリフ）\n\n' if SCENE_COUNT >= 5 else ''}{'シーン6（CTA）:\n（セリフ）\n\n' if SCENE_COUNT >= 6 else ''}
[キーワード]
各シーンのPexels画像検索用英語キーワード（1〜3語）:
scene1: （英語キーワード）
scene2: （英語キーワード）
scene3: （英語キーワード）
scene4: （英語キーワード）
{'scene5: （英語キーワード）' if SCENE_COUNT >= 5 else ''}{'\nscene6: （英語キーワード）' if SCENE_COUNT >= 6 else ''}

[秒数配分]
scene1: （秒数）
scene2: （秒数）
scene3: （秒数）
scene4: （秒数）
{'scene5: （秒数）' if SCENE_COUNT >= 5 else ''}{'\nscene6: （秒数）' if SCENE_COUNT >= 6 else ''}
"""
raw_script = call_gemini(script_prompt)

# パース
script_text  = (re.search(r'\[台本\]([\s\S]*?)(?=\[キーワード\])', raw_script) or ['',''])[1] if re.search(r'\[台本\]', raw_script) else raw_script
kw_block     = re.search(r'\[キーワード\]([\s\S]*?)(?=\[秒数配分\])', raw_script)
time_block   = re.search(r'\[秒数配分\]([\s\S]*?)$', raw_script)

keywords = []
if kw_block:
    for line in kw_block.group(1).split('\n'):
        m = re.match(r'scene\d+:\s*(.+)', line.strip(), re.I)
        if m: keywords.append(m.group(1).strip())
while len(keywords) < SCENE_COUNT:
    keywords.append('fitness motivation')

timings = []
if time_block:
    for line in time_block.group(1).split('\n'):
        m = re.match(r'scene\d+:\s*(\d+)', line.strip(), re.I)
        if m: timings.append(int(m.group(1)))
while len(timings) < SCENE_COUNT:
    timings.append(DURATION // SCENE_COUNT)

# シーンごとのセリフを抽出
scene_lines = re.split(r'シーン\d+[（(][^)）]*[)）]\s*[:：]', script_text)
scene_texts = [s.strip() for s in scene_lines[1:] if s.strip()]
while len(scene_texts) < SCENE_COUNT:
    scene_texts.append(THEME)

print(f'  ✓ 台本生成完了（{SCENE_COUNT}シーン）')
print()
print('=== 生成された台本 ===')
print(script_text[:800])

In [ ]:
# ════════════════════════════════════════════════════════
# セル5: Pexelsから著作権フリー画像を自動取得
# ════════════════════════════════════════════════════════

import time
from pathlib import Path
from datetime import datetime

ts          = datetime.now().strftime('%Y%m%d_%H%M%S')
safe_theme  = re.sub(r'[\\/:*?"<>|]', '_', THEME)[:25]
PROJECT_DIR = Path(f'/content/drive/MyDrive/YouTube_Production/{ts}_{safe_theme}')
IMG_DIR     = PROJECT_DIR / 'images'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

(PROJECT_DIR / '台本.txt').write_text(script_text, encoding='utf-8')

print('[3/5] 著作権フリー画像を取得中...')

def search_pexels(keyword):
    res = _req.get(
        'https://api.pexels.com/v1/search',
        headers={'Authorization': PEXELS_API_KEY},
        params={'query': keyword, 'per_page': 1, 'orientation': 'portrait'},
        timeout=15
    )
    if res.status_code != 200: return None
    photos = res.json().get('photos', [])
    return photos[0] if photos else None

scenes = []
for i, (kw, dur) in enumerate(zip(keywords[:SCENE_COUNT], timings[:SCENE_COUNT])):
    time.sleep(0.6)
    filename = f'scene_{i+1:02d}.jpg'
    img_path = IMG_DIR / filename

    photo = search_pexels(kw)
    if photo:
        img_url = photo['src'].get('portrait') or photo['src'].get('large')
        img_res = _req.get(img_url, timeout=30)
        if img_res.status_code == 200:
            img_path.write_bytes(img_res.content)
            print(f'  ✓ シーン{i+1}: {kw} → {filename}')
        else:
            print(f'  ⚠ シーン{i+1}: 画像DL失敗')
    else:
        print(f'  ⚠ シーン{i+1}: 「{kw}」の画像が見つかりません')

    scenes.append({
        'scene':    i + 1,
        'image':    f'images/{filename}',
        'keyword':  kw,
        'duration': dur,
        'effect':   'zoom_in' if i % 2 == 0 else 'zoom_out',
        'text':     scene_texts[i] if i < len(scene_texts) else '',
    })

config = {
    'theme': THEME,
    'target_duration': DURATION,
    'scene_count': SCENE_COUNT,
    'scenes': scenes,
    'video_settings': {'width': 1080, 'height': 1920, 'fps': 30, 'format': 'mp4'},
    'created_at': datetime.now().isoformat(),
}
(PROJECT_DIR / 'production_config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'\n✅ 画像取得完了 → {PROJECT_DIR.name}')

In [ ]:
# ════════════════════════════════════════════════════════
# セル6: gTTSで音声を自動生成
# ════════════════════════════════════════════════════════

import subprocess, tempfile
from gtts import gTTS

TMP_DIR = Path(tempfile.mkdtemp(prefix='yt_shorts_'))

def clean_text(text):
    text = re.sub(r'\[速く\]|\[ゆっくり\]|\[強調\]', '', text)
    text = re.sub(r'\[間\d+\.?\d*\]', '、', text)
    text = re.sub(r'（ここにセリフ）|\(ここにセリフ\)', '', text)
    text = re.sub(r'シーン\d+[（(][^)）]*[)）]\s*[:：]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def run_ff(*args):
    r = subprocess.run(['ffmpeg', '-y', *[str(a) for a in args]],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-1000:])

print('[4/5] 音声を生成中（gTTS）...')

wav_parts = []
for i, scene in enumerate(scenes):
    text  = clean_text(scene.get('text', THEME))
    if not text or len(text) < 2:
        text = THEME
    mp3 = TMP_DIR / f'voice_{i+1:02d}.mp3'
    wav = TMP_DIR / f'voice_{i+1:02d}.wav'
    try:
        gTTS(text=text, lang='ja').save(str(mp3))
        run_ff('-i', mp3, '-ar', '44100', '-ac', '1', wav)
        wav_parts.append(wav)
        print(f'  ✓ シーン{i+1}: 音声生成完了')
    except Exception as e:
        print(f'  ⚠ シーン{i+1}: {e}')

VOICE_AAC = None
if wav_parts:
    list_file = TMP_DIR / 'voice_list.txt'
    list_file.write_text('\n'.join(f"file '{p}'" for p in wav_parts))
    combined  = TMP_DIR / 'voice_combined.wav'
    voice_aac = TMP_DIR / 'voice.aac'
    run_ff('-f', 'concat', '-safe', '0', '-i', list_file, '-c', 'copy', combined)
    run_ff('-i', combined, '-c:a', 'aac', '-ar', '44100', voice_aac)
    VOICE_AAC = voice_aac
    print('\n  ✅ 全シーン音声結合完了')
else:
    print('  ⚠ 音声生成なし（映像のみで続行）')

In [ ]:
# ════════════════════════════════════════════════════════
# セル7: ケン・バーンズ動画クリップを生成して最終合成
# ════════════════════════════════════════════════════════

W, H, FPS = 1080, 1920, 30

def make_clip(img_path, dur, out, effect='zoom_in'):
    frames    = int(dur * FPS)
    step      = 0.15 / max(frames, 1)
    zoom_expr = f"min(1+{step:.6f}*on,1.15)" if effect == 'zoom_in' else f"if(eq(on,1),1.15,max(1.0,zoom-{step:.6f}))"
    zoompan   = f"zoompan=z='{zoom_expr}':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={frames}:s={W}x{H}:fps={FPS}"
    scale_pad = f"scale={W}:{H}:force_original_aspect_ratio=increase,crop={W}:{H}"
    run_ff('-loop','1','-i',img_path,'-vf',f'{scale_pad},{zoompan}',
           '-t',str(dur),'-an','-c:v','libx264','-preset','fast','-crf','22','-pix_fmt','yuv420p', out)

def ass_time(s):
    return f'{int(s//3600)}:{int((s%3600)//60):02d}:{int(s%60):02d}.{int((s%1)*100):02d}'

print('[5/5] 動画を生成中...')

clip_paths = []
for scene in scenes:
    img  = PROJECT_DIR / scene['image']
    out  = TMP_DIR / f"clip_{scene['scene']:02d}.mp4"
    dur  = scene['duration']
    eff  = scene['effect']
    if img.exists():
        make_clip(img, dur, out, eff)
        print(f"  ✓ シーン{scene['scene']}: {eff}")
    else:
        run_ff('-f','lavfi','-i',f'color=black:s={W}x{H}:r={FPS}','-t',str(dur),
               '-c:v','libx264','-preset','fast', out)
        print(f"  ⚠ シーン{scene['scene']}: 画像なし → 黒画面")
    clip_paths.append(out)

# クリップ結合
list_file   = TMP_DIR / 'clip_list.txt'
list_file.write_text('\n'.join(f"file '{p}'" for p in clip_paths))
merged      = TMP_DIR / 'merged.mp4'
run_ff('-f','concat','-safe','0','-i',list_file,'-c','copy', merged)

# テロップ（ASSサブタイトル）
ass_header = f"""[Script Info]\nPlayResX:{W}\nPlayResY:{H}\nScriptType:v4.00+\n\n[V4+ Styles]\nFormat:Name,Fontname,Fontsize,PrimaryColour,OutlineColour,Bold,Outline,Shadow,Alignment,MarginV\nStyle:Default,Arial,72,&H00FFFFFF,&H00000000,-1,4,1,2,{int(H*0.18)}\n\n[Events]\nFormat:Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n"""
events = []
t = 0.0
for scene in scenes:
    d = scene['duration']
    kw = scene['keyword'].replace(',', ' ')[:20]
    events.append(f"Dialogue:0,{ass_time(t)},{ass_time(t+d)},Default,,0,0,0,,{kw}")
    t += d
subs = TMP_DIR / 'subs.ass'
subs.write_text(ass_header + '\n'.join(events), encoding='utf-8')
subs_esc = str(subs).replace('\\', '/').replace(':', '\\:')

# 最終合成
output_dir = PROJECT_DIR / 'output'
output_dir.mkdir(exist_ok=True)
out_ts   = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_PATH = output_dir / f'shorts_{out_ts}.mp4'

if VOICE_AAC and VOICE_AAC.exists():
    run_ff('-i', merged, '-i', VOICE_AAC,
           '-vf', f'ass={subs_esc}',
           '-map', '0:v', '-map', '1:a',
           '-c:v', 'libx264', '-preset', 'medium', '-crf', '20',
           '-c:a', 'aac', '-b:a', '192k',
           '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
           '-t', str(DURATION), OUT_PATH)
else:
    run_ff('-i', merged,
           '-vf', f'ass={subs_esc}',
           '-c:v', 'libx264', '-preset', 'medium', '-crf', '20',
           '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
           '-t', str(DURATION), OUT_PATH)

size_mb = OUT_PATH.stat().st_size / 1_048_576
print(f'\n{"="*50}')
print(f'  ✅ 動画完成！')
print(f'  テーマ  : {THEME}')
print(f'  保存先  : {OUT_PATH}')
print(f'  サイズ  : {size_mb:.1f} MB')
print(f'{"="*50}')

In [ ]:
# ════════════════════════════════════════════════════════
# セル8: 完成動画をここで再生確認
# ════════════════════════════════════════════════════════

from IPython.display import Video, display

# Colabで直接再生するためにローカルにコピー
import shutil
local_preview = '/content/preview.mp4'
shutil.copy(str(OUT_PATH), local_preview)

print(f'▶ 動画プレビュー（縦型なので縦に表示されます）')
print(f'  Googleドライブの保存先: YouTube_Production/{PROJECT_DIR.name}/output/')
display(Video(local_preview, width=360))